## **LangChain**

 <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>

In [24]:
import dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate

### 1 & 2. Model and Chat Model 

In [6]:
Model_API = dotenv.get_key("../.env", "GEMINI_API")

In [7]:
def get_chat_llm (params=None):
    
    llm = ChatGoogleGenerativeAI(
        model = "gemini-3-flash-preview",
        temperature=0.3,
        api_key = Model_API
    )
    return llm

In [8]:
llm_model = get_chat_llm()

### 3. Chat Message 

In [9]:
message = llm_model.invoke([
    SystemMessage("You are a helpful assistant to find best car based on the user requirement in a one short sentence"),
    HumanMessage("I am more interested about sport sedan. please suggest me a one")
])

In [10]:
print(message)

content=[{'type': 'text', 'text': 'The BMW M3 is the definitive benchmark for performance, handling, and luxury in the sport sedan category.', 'extras': {'signature': 'EpMJCpAJAQw51sd9WM7hZr8feAjpg+EVp769bTgtICDhFMbRI+XUb5DXMSuKzmLAat6ydpjjWerT4Fxv2HZhd0q3+0GadKvZdCS2hR3bFIn9hbq8cXPh/nvNGWMDZweybwospGhRf/J/ChN0RBtC+QRkiUVYbxulKFOfDFTki+Vr0IluzcXjEk32k27dyrWe4b5izmpt07lo7TjMUunYQdu04tnK6z50ICeo/XGvn3+KgsGGXZjQgpdL5973ES0h3/+LhYBWSd1MbykWnQFlOlGWdLKa4C7f3pCbU8ZD2U+aB2Yz/xZuGJ+n0VBvTUh/RPEzvRAKsQj1C+JpWM996AkF5AXLeJ2IGGvo66QCT13kWSyVfU6Ugr/nTtosQgqEGvClWHrt0SqQWHk42oNq+458YMNmjDud4GYm6el1is5ZmVKP28DKSD7abGaubmmTt2K/m0LJXYHB10vC+/KTjkX/GFn63ENmasqVI43pHY4Uud2U2mBPPfyupPhYma2mMHV+t/oLFHxVm6vlUKomo19uCMlzoD1X8bBQ5vsTLsR/VnYUbvMrrBdAtDDtY5rdHgm2kvo4kfl6av+xmfFgP1vauT2nnb6ilsPpH5VxJ/exadtY6vjDlx/8GGUT8Q8Ls3q0my4Ywn5W4YxCkvCG12BonrTqQLU2kc6C67sDo/xfNKOfZ0NO7y6nU0rDfi/XWFoZ+gTCpWWSN8y0UrhtluX614VPC9GHACAGsOCy3GrfuEDGF+vQp2KjgYf6I13P5Tn7h27vzFpMaqURplgP3T3kHKUX+XAGH6KFw1M/flYduPO4DPKup6KpFjaZHCuz

### 4. Prompt templates

We use prompt templates to give single or multi messages to the model to get more clear and accurate reponse for our questions.

##### Promt Template

**Prompt Template** uses to give single string input for the model as a message 

In [11]:
prompt = PromptTemplate.from_template("Tell me a {type} about the {topic}")
input_ = {"type":"joke", "topic":"computer"}

prompt.invoke(input_)

StringPromptValue(text='Tell me a joke about the computer')

In [12]:
output = prompt | llm_model
output

PromptTemplate(input_variables=['topic', 'type'], input_types={}, partial_variables={}, template='Tell me a {type} about the {topic}')
| ChatGoogleGenerativeAI(profile={'name': 'Gemini 3 Flash Preview', 'release_date': '2025-12-17', 'last_updated': '2025-12-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-3-flash-preview', temperature=0.3, client=<google.genai.client.Client object at 0x000001F52728F390>, default_metadata=(), model_kwargs={})

##### Chat Prompt Template

These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [13]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "tell me a joke about {topic}")
])

input_ = {"topic":"cat"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cat', additional_kwargs={}, response_metadata={})])

### 5. Message PlaceHolder

Message Placeholder, we use to apply relevant list of messages to a paticular positions in the chat templates.

In [16]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to user"),
    MessagesPlaceholder("msgs")
])

input_ = {
    "msgs": [
        HumanMessage(content='What is the world largest mountain?'),
        HumanMessage(content="Please give me a short description about toyota supra")
    ]
}

formatted_prompt = chat_prompt.invoke(input_)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant to user', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the world largest mountain?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please give me a short description about toyota supra', additional_kwargs={}, response_metadata={})])

In [17]:
response = llm_model.invoke(formatted_prompt)

In [21]:
print(response.content[0]['text'])

The answer to your questions depends on how you measure "largest," so here are the two main contenders for the world's largest mountain:

1.  **Mount Everest:** This is the **highest** mountain in the world, reaching **29,032 feet (8,849 meters)** above sea level.
2.  **Mauna Kea:** This is the **tallest** mountain in the world when measured from base to peak. While it only stands 13,803 feet above sea level, its base sits on the ocean floor; from the very bottom to the top, it measures over **33,500 feet (10,210 meters)**.

***

### Toyota Supra: A Short Description
The **Toyota Supra** is an iconic Japanese sports car and grand tourer that first debuted in 1978. It gained legendary status primarily through its fourth generation (the "A80" or Mark IV), produced in the 1990s, which became famous for its incredibly durable **2JZ-GTE engine** and its starring role in the *Fast & Furious* film franchise.

After a 17-year hiatus, Toyota revived the nameplate in 2019 with the **GR Supra (A9

In [22]:
chain = chat_prompt | llm_model
response = chain.invoke(input_)
print(response.content[0]['text'])

The answer to your first question depends on how you define "largest":

*   **Highest Altitude (Above Sea Level):** **Mount Everest**, located in the Himalayas on the border of Nepal and China. Its peak is 29,031.7 feet (8,848.86 meters) above sea level.
*   **Tallest (Base to Peak):** **Mauna Kea** in Hawaii. While its peak is only 13,803 feet above sea level, its base sits on the ocean floor. From the very bottom to the top, it measures about 33,500 feet (10,210 meters).
*   **Largest by Mass/Volume:** **Mauna Loa** in Hawaii. It is the world's largest shield volcano, covering a massive area and containing approximately 18,000 cubic miles of rock.

***

### Toyota Supra: A Short Description
The **Toyota Supra** is an iconic Japanese sports car and grand tourer that first debuted in 1978. While it began as a high-performance version of the Toyota Celica, it eventually became its own distinct model.

The Supra gained legendary status primarily through its **fourth generation (A80)**, p

### 6. Example Selector

If you want to provide your model with examples and you have a large number of them, you need to filter out and select the most relevant examples for a given query. This task is performed by **Example Selectors**.

There are main examples selectors:

* `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
* `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
* `Length`: Selects examples based on how many can fit within a certain length
* `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choo

In [25]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input":"windy", "output":"calm"}
]
example_prompt = PromptTemplate(
    template= "input: {input}\noutput: {output}",
    input_variables=['input', 'output']
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25
)

## for example driven template, we use fewshot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    prefix = "Give the antonym of every input",
    suffix = "input: {input}\noutput:",
    input_variables = ['input']
)

In [26]:
print(dynamic_prompt.format(input = "big"))

Give the antonym of every input

input: happy
output: sad

input: tall
output: short

input: energetic
output: lethargic

input: sunny
output: gloomy

input: windy
output: calm

input: big
output:


In [27]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(input = long_string))

Give the antonym of every input

input: happy
output: sad

input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
output:
